# Stage 2: Coupled SSM on Revised Stage 1 SST Outputs

Run this notebook after the revised Stage 1 notebook has created dataset-specific `stage1_turns_embedded.csv` files under `/content/optimal-nego/outputs/`.

In [ ]:
!pip install -q numpy pandas scipy scikit-learn

In [ ]:
# ============================================================
# STAGE 2: Coupled SSM on Stage 1 SST / GoEmotions trajectories
# Revised for multi-file Stage 1 outputs
# ============================================================
#
# Expected Stage 1 directory layout:
# /content/optimal-nego/outputs/
#   stage1_manifest.csv
#   dealerships_nego/
#       stage1_turns_embedded.csv
#       stage1_sst_dimensions.csv
#       stage1_report.txt
#   emaad_sales/
#       stage1_turns_embedded.csv
#       ...
#   heddaya_nego/
#       stage1_turns_embedded.csv
#       ...
#
# Main changes from the older Stage 2 script:
# 1. Does NOT reread raw data or rebuild lexicon-based sparse emotion features.
# 2. Reads Stage 1 retained SST dimensions directly from stage1_turns_embedded.csv.
# 3. Runs independently for each dataset folder produced by Stage 1.
# 4. Handles variable SST dimensionality p per dataset.
# 5. Skips outcome prediction when outcome is missing or has only one class.
# 6. Writes Stage 2 outputs inside each dataset-specific output folder.
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

# ----------------------------
# Paths
# ----------------------------
BASE_DIR = Path("/content/optimal-nego")
OUTPUT_ROOT = BASE_DIR / "outputs"
MANIFEST_PATH = OUTPUT_ROOT / "stage1_manifest.csv"

# Datasets expected from the revised Stage 1 notebook.
# If the manifest exists, it is used as the source of truth.
DATASET_NAMES = ["dealerships_nego", "emaad_sales", "heddaya_nego"]

# Model-selection settings
K_GRID = [2, 3, 4]
MAX_ITER = 60
TOL = 0.5
RANDOM_SEED = 42

# Optional smoothing of Stage 1 probabilities before SSM.
# Stage 1 already produces dense model-based probabilities; keep a light centered rolling mean.
SMOOTH_WINDOW = 3

# Metadata columns in stage1_turns_embedded.csv.
META_COLS = {
    "global_row_id", "dataset_name", "source_file",
    "conversation_id", "turn_index", "speaker_id", "role",
    "start_time", "end_time", "duration_min", "text", "outcome",
}

print("=" * 70)
print("STAGE 2: COUPLED SSM ON STAGE 1 SST EMOTIONAL TRAJECTORIES")
print("=" * 70)
print("Output root:", OUTPUT_ROOT)


# ----------------------------
# Helpers
# ----------------------------
def normalize_outcome(value):
    """Map outcome labels to binary values where possible."""
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower()

    sale_terms = {
        "sale", "sold", "deal", "closed", "success", "yes", "1", "true",
        "positive", "purchase", "bought",
    }
    no_sale_terms = {
        "no sale", "nosale", "no_sale", "no-deal", "no deal", "failed",
        "failure", "no", "0", "false", "negative", "not sold",
    }

    if s in sale_terms:
        return 1
    if s in no_sale_terms:
        return 0

    # More forgiving pattern checks.
    if "no" in s and ("sale" in s or "deal" in s):
        return 0
    if "sale" in s or "sold" in s or "deal" in s:
        return 1

    return np.nan


def load_dataset_list():
    """Load dataset names from Stage 1 manifest when available."""
    if MANIFEST_PATH.exists():
        manifest = pd.read_csv(MANIFEST_PATH)
        if "dataset_name" in manifest.columns:
            names = manifest["dataset_name"].dropna().astype(str).tolist()
            if names:
                return names
    return DATASET_NAMES


def find_stage1_file(dataset_name):
    """Return path to dataset-specific stage1_turns_embedded.csv."""
    direct = OUTPUT_ROOT / dataset_name / "stage1_turns_embedded.csv"
    if direct.exists():
        return direct

    # Fallback: search recursively.
    matches = list(OUTPUT_ROOT.glob(f"**/{dataset_name}/stage1_turns_embedded.csv"))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Could not find Stage 1 embedded file for {dataset_name}. "
        f"Expected: {direct}"
    )


def get_emotion_columns(df):
    """Infer retained SST emotion columns from Stage 1 embedded file."""
    numeric_cols = []
    for col in df.columns:
        if col in META_COLS:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)

    if not numeric_cols:
        raise ValueError(
            "No numeric SST emotion columns found. "
            "Check that stage1_turns_embedded.csv contains retained dimensions."
        )

    return numeric_cols


def sort_stage1_turns(df):
    """Sort turns robustly using start_time when available, otherwise turn_index."""
    df = df.copy()
    df["conversation_id"] = df["conversation_id"].astype(str)

    if "turn_index" not in df.columns:
        df["turn_index"] = df.groupby("conversation_id").cumcount()

    if "start_time" in df.columns and df["start_time"].notna().any():
        return df.sort_values(["conversation_id", "start_time", "turn_index"]).reset_index(drop=True)

    return df.sort_values(["conversation_id", "turn_index"]).reset_index(drop=True)


def smooth_emotions(df, emotion_cols, window=3):
    """Apply light within-conversation rolling smoothing to dense Stage 1 emotion probabilities."""
    df = df.copy()
    if window is None or window <= 1:
        return df

    for col in emotion_cols:
        df[col] = (
            df.groupby("conversation_id")[col]
            .transform(lambda x: x.rolling(window, min_periods=1, center=True).mean())
        )
    return df


def build_coupled_sequences(df, emotion_cols):
    """
    Build coupled buyer/seller observations.

    At each turn t, the observation is:
        [latest buyer SST vector, latest seller SST vector]

    If role is unknown, neither vector is updated at that turn.
    """
    conv_ids = sorted(df["conversation_id"].astype(str).unique())
    p = len(emotion_cols)
    conv_data = {}

    for cid in conv_ids:
        grp = df[df["conversation_id"].astype(str) == str(cid)].copy()
        grp = sort_stage1_turns(grp)

        T = len(grp)
        buyer_series = np.zeros((T, p))
        seller_series = np.zeros((T, p))

        last_buyer = np.zeros(p)
        last_seller = np.zeros(p)

        for t, (_, row) in enumerate(grp.iterrows()):
            ev = row[emotion_cols].to_numpy(dtype=float)
            role = str(row.get("role", "unknown")).strip().lower()

            if role == "buyer":
                last_buyer = ev
            elif role == "seller":
                last_seller = ev

            buyer_series[t] = last_buyer
            seller_series[t] = last_seller

        X = np.hstack([buyer_series, seller_series])

        if "outcome" in grp.columns:
            raw_outcome = grp["outcome"].dropna()
            raw_outcome = raw_outcome.iloc[0] if len(raw_outcome) else np.nan
        else:
            raw_outcome = np.nan

        conv_data[str(cid)] = {
            "X": X,
            "T": T,
            "raw_outcome": raw_outcome,
            "outcome_binary": normalize_outcome(raw_outcome),
        }

    sequences = [conv_data[cid]["X"] for cid in conv_ids]
    return conv_ids, conv_data, sequences


# ----------------------------
# Kalman EM
# ----------------------------
def run_em(sequences, k, p, max_iter=60, tol=0.5, seed=42):
    """Estimate a linear Gaussian state-space model with Kalman EM."""
    rng = np.random.default_rng(seed + k)
    eps = 1e-6

    A = np.eye(k) * 0.9 + rng.normal(0, 0.01, size=(k, k))
    C = rng.normal(0, 0.05, size=(p, k))
    Q = np.eye(k) * 0.1
    R = np.eye(p) * 0.3
    mu0 = np.zeros(k)
    V0 = np.eye(k)
    prev_ll = -np.inf

    for itr in range(max_iter):
        T_tot = 0
        s_zz = np.zeros((k, k))
        s_zt_zt1 = np.zeros((k, k))
        s_zt1_zt1 = np.zeros((k, k))
        s_xz = np.zeros((p, k))
        s_xx = np.zeros((p, p))
        mu0_sum = np.zeros(k)
        V0_sum = np.zeros((k, k))
        total_ll = 0.0
        all_smoothed = []

        for X in sequences:
            T = len(X)
            if T == 0:
                continue

            mu_f = np.zeros((T, k))
            V_f = np.zeros((T, k, k))
            mu_p = np.zeros((T, k))
            V_p = np.zeros((T, k, k))

            # Forward filter
            for t in range(T):
                mp = mu0 if t == 0 else A @ mu_f[t - 1]
                Vp = V0 if t == 0 else A @ V_f[t - 1] @ A.T + Q
                Vp = 0.5 * (Vp + Vp.T) + eps * np.eye(k)

                mu_p[t] = mp
                V_p[t] = Vp

                S = C @ Vp @ C.T + R
                S = 0.5 * (S + S.T) + eps * np.eye(p)
                Si = np.linalg.pinv(S)
                K = Vp @ C.T @ Si
                innovation = X[t] - C @ mp

                mu_f[t] = mp + K @ innovation
                V_f[t] = (np.eye(k) - K @ C) @ Vp
                V_f[t] = 0.5 * (V_f[t] + V_f[t].T) + eps * np.eye(k)

                sign, logdet = np.linalg.slogdet(S)
                if sign > 0:
                    total_ll -= 0.5 * (
                        logdet + innovation @ Si @ innovation + p * np.log(2 * np.pi)
                    )

            # Rauch-Tung-Striebel smoother
            mu_s = mu_f.copy()
            V_s = V_f.copy()
            V_cross = np.zeros((max(T - 1, 0), k, k))

            for t in range(T - 2, -1, -1):
                G = V_f[t] @ A.T @ np.linalg.pinv(V_p[t + 1])
                mu_s[t] = mu_f[t] + G @ (mu_s[t + 1] - mu_p[t + 1])
                V_s[t] = V_f[t] + G @ (V_s[t + 1] - V_p[t + 1]) @ G.T
                V_s[t] = 0.5 * (V_s[t] + V_s[t].T) + eps * np.eye(k)
                V_cross[t] = V_s[t + 1] @ G.T

            all_smoothed.append(mu_s)

            T_tot += T
            mu0_sum += mu_s[0]
            V0_sum += V_s[0] + np.outer(mu_s[0], mu_s[0])

            for t in range(T):
                ez = mu_s[t]
                ezz = V_s[t] + np.outer(ez, ez)
                s_zz += ezz
                s_xz += np.outer(X[t], ez)
                s_xx += np.outer(X[t], X[t])

            for t in range(T - 1):
                s_zt_zt1 += V_cross[t] + np.outer(mu_s[t + 1], mu_s[t])
                s_zt1_zt1 += V_s[t] + np.outer(mu_s[t], mu_s[t])

        n_seq = len(sequences)
        mu0 = mu0_sum / max(n_seq, 1)
        V0 = V0_sum / max(n_seq, 1) - np.outer(mu0, mu0)
        V0 = 0.5 * (V0 + V0.T) + eps * np.eye(k)

        A = s_zt_zt1 @ np.linalg.pinv(s_zt1_zt1 + eps * np.eye(k))
        Q_raw = (s_zz - A @ s_zt_zt1.T) / max(T_tot, 1)
        Q = 0.5 * (Q_raw + Q_raw.T) + eps * np.eye(k)

        C = s_xz @ np.linalg.pinv(s_zz + eps * np.eye(k))
        R_raw = (s_xx - C @ s_xz.T) / max(T_tot, 1)
        R_diag = np.maximum(np.diag(R_raw), eps)
        R = np.diag(R_diag)

        delta = total_ll - prev_ll
        if itr < 3 or itr % 10 == 0:
            print(f"    iter {itr + 1:3d}: ll={total_ll:.1f}  Δ={delta:.1f}")

        if itr > 0 and abs(delta) < tol:
            print(f"    Converged at iter {itr + 1}")
            break

        prev_ll = total_ll

    T_tot_all = sum(len(X) for X in sequences)
    n_params = k * k + p * k + k * (k + 1) // 2 + p
    bic = -2 * total_ll + n_params * np.log(max(T_tot_all, 1))
    spectral_radius = float(max(abs(np.linalg.eigvals(A))))

    return {
        "A": A,
        "C": C,
        "Q": Q,
        "R": R,
        "mu0": mu0,
        "V0": V0,
        "ll": total_ll,
        "bic": bic,
        "sr": spectral_radius,
        "smoothed": all_smoothed,
    }


def save_latent_states(dataset_out, conv_ids, conv_data, smoothed, best_k):
    rows = []
    for cid, z_seq in zip(conv_ids, smoothed):
        for t, z in enumerate(z_seq):
            row = {
                "conversation_id": cid,
                "turn": t,
                "raw_outcome": conv_data[cid]["raw_outcome"],
                "outcome_binary": conv_data[cid]["outcome_binary"],
            }
            for d in range(best_k):
                row[f"z_{d + 1}"] = round(float(z[d]), 6)
            rows.append(row)

    latent_df = pd.DataFrame(rows)
    path = dataset_out / "stage2_latent_states.csv"
    latent_df.to_csv(path, index=False)
    print(f"  stage2_latent_states.csv: {latent_df.shape}")
    return latent_df


def run_outcome_prediction(dataset_out, conv_ids, conv_data, smoothed, best_k, emotion_dim):
    """Predict binary outcomes if available."""
    y = np.array([conv_data[cid]["outcome_binary"] for cid in conv_ids], dtype=float)
    valid = ~np.isnan(y)

    if valid.sum() < 10 or len(np.unique(y[valid])) < 2:
        print("  Outcome prediction skipped: missing outcome or only one outcome class.")
        empty = pd.DataFrame({
            "conversation_id": conv_ids,
            "raw_outcome": [conv_data[cid]["raw_outcome"] for cid in conv_ids],
            "outcome_binary": y,
            "pred_prob": np.nan,
        })
        empty.to_csv(dataset_out / "stage2_outcome_prediction.csv", index=False)
        return None

    feats = []
    valid_conv_ids = []
    valid_y = []

    for idx, cid in enumerate(conv_ids):
        if not valid[idx]:
            continue

        z_seq = smoothed[idx]
        T = len(z_seq)

        z_mean = z_seq.mean(axis=0)
        z_std = z_seq.std(axis=0) + 1e-8
        z_late = z_seq[max(0, 2 * T // 3):].mean(axis=0)

        if T >= 2:
            slopes = np.array([
                np.polyfit(np.arange(T), z_seq[:, d], 1)[0]
                for d in range(best_k)
            ])
        else:
            slopes = np.zeros(best_k)

        X_obs = conv_data[cid]["X"]
        # Mean absolute buyer-seller divergence by retained SST dimension.
        div = np.abs(X_obs[:, :emotion_dim] - X_obs[:, emotion_dim:]).mean(axis=0)

        feats.append(np.concatenate([z_mean, z_std, z_late, slopes, div]))
        valid_conv_ids.append(cid)
        valid_y.append(int(y[idx]))

    F = np.asarray(feats)
    y_valid = np.asarray(valid_y)

    # Use a safe number of folds based on the minority class count.
    _, counts = np.unique(y_valid, return_counts=True)
    n_splits = min(5, int(counts.min()))

    if n_splits < 2:
        print("  Outcome prediction skipped: not enough examples per class for CV.")
        return None

    F_sc = StandardScaler().fit_transform(F)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=0.1)

    aucs = []
    preds = np.zeros(len(y_valid))

    for fold, (tr, te) in enumerate(cv.split(F_sc, y_valid)):
        clf.fit(F_sc[tr], y_valid[tr])
        prob = clf.predict_proba(F_sc[te])[:, 1]
        preds[te] = prob
        auc = roc_auc_score(y_valid[te], prob)
        aucs.append(auc)
        print(f"  Fold {fold + 1}: AUC={auc:.4f}")

    mean_auc = float(np.mean(aucs))
    overall_auc = float(roc_auc_score(y_valid, preds))

    result_df = pd.DataFrame({
        "conversation_id": valid_conv_ids,
        "outcome": y_valid,
        "pred_prob": preds,
    })
    result_df.to_csv(dataset_out / "stage2_outcome_prediction.csv", index=False)

    return {
        "n_conversations_used": int(len(y_valid)),
        "n_splits": int(n_splits),
        "mean_cv_auc": mean_auc,
        "overall_auc": overall_auc,
    }


def run_stage2_for_dataset(dataset_name):
    dataset_out = OUTPUT_ROOT / dataset_name
    dataset_out.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    embedded_path = find_stage1_file(dataset_name)
    df = pd.read_csv(embedded_path)
    df = sort_stage1_turns(df)

    emotion_cols = get_emotion_columns(df)
    p_emotion = len(emotion_cols)
    obs_dim = 2 * p_emotion

    print("[Step 1] Loading Stage 1 embedded SST turns...")
    print(f"  File                 : {embedded_path}")
    print(f"  Turns                : {len(df):,}")
    print(f"  Conversations         : {df['conversation_id'].nunique():,}")
    print(f"  Retained SST dims     : {p_emotion}")
    print(f"  Observation dimension : {obs_dim} = buyer({p_emotion}) + seller({p_emotion})")

    # Coerce emotion cols to numeric and smooth.
    for col in emotion_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    df = smooth_emotions(df, emotion_cols, window=SMOOTH_WINDOW)
    df.to_csv(dataset_out / "stage2_trajectories.csv", index=False)

    E_all = df[emotion_cols].to_numpy(dtype=float)
    sparsity = float((E_all <= 1e-12).mean() * 100)
    active = float((E_all > 1e-12).sum(axis=1).mean())

    print("[Step 2] Building coupled buyer-seller observations...")
    conv_ids, conv_data, sequences = build_coupled_sequences(df, emotion_cols)
    avg_T = float(np.mean([len(s) for s in sequences])) if sequences else 0.0

    print(f"  Conversations         : {len(sequences):,}")
    print(f"  Avg turns/conversation: {avg_T:.1f}")
    print(f"  Dense sparsity        : {sparsity:.1f}%")
    print(f"  Active dims/turn      : {active:.2f}")

    if len(sequences) < 2:
        raise ValueError(f"Not enough conversations for Stage 2 in {dataset_name}.")

    print(f"[Step 3] BIC-based latent dimension selection k={K_GRID}...")
    bic_results = {}
    model_cache = {}

    for k in K_GRID:
        print(f"\n  --- k={k} ---")
        res = run_em(
            sequences=sequences,
            k=k,
            p=obs_dim,
            max_iter=MAX_ITER,
            tol=TOL,
            seed=RANDOM_SEED,
        )
        bic_results[k] = float(res["bic"])
        model_cache[k] = res
        print(
            f"  k={k}: ll={res['ll']:.1f}  "
            f"BIC={res['bic']:.1f}  spectral_r={res['sr']:.4f}"
        )

    best_k = min(bic_results, key=bic_results.get)
    best = model_cache[best_k]

    print(f"\n[Step 4] Final model: k={best_k}, spectral_r={best['sr']:.4f}")

    np.savez(
        dataset_out / "stage2_ssm_params.npz",
        A=best["A"],
        C=best["C"],
        Q=best["Q"],
        R=best["R"],
        mu0=best["mu0"],
        V0=best["V0"],
        best_k=best_k,
        emotion_cols=np.array(emotion_cols, dtype=object),
    )

    with open(dataset_out / "stage2_bic_results.json", "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in bic_results.items()}, f, indent=2)

    print("\n[Step 5] Saving latent state trajectories...")
    latent_df = save_latent_states(dataset_out, conv_ids, conv_data, best["smoothed"], best_k)

    print("\n[Step 6] Outcome prediction from latent trajectories...")
    pred_summary = run_outcome_prediction(
        dataset_out=dataset_out,
        conv_ids=conv_ids,
        conv_data=conv_data,
        smoothed=best["smoothed"],
        best_k=best_k,
        emotion_dim=p_emotion,
    )

    diag_lines = [
        "STAGE 2 DIAGNOSTICS",
        "=" * 50,
        f"Dataset                    : {dataset_name}",
        f"Stage 1 embedded file       : {embedded_path}",
        f"Turns                       : {len(df)}",
        f"Conversations               : {len(sequences)}",
        f"Retained SST dimensions     : {p_emotion}",
        f"Observation dimension       : {obs_dim}",
        f"Dense trajectory sparsity   : {sparsity:.1f}%",
        f"Active dims per turn        : {active:.2f}",
        "",
        "Retained emotion dimensions:",
        "  " + ", ".join(emotion_cols),
        "",
        "BIC model selection:",
    ]

    for k, v in sorted(bic_results.items()):
        diag_lines.append(f"  k={k}: BIC={v:.1f}" + (" <- best" if k == best_k else ""))

    diag_lines += [
        "",
        f"Best k                      : {best_k}",
        f"Final log-likelihood        : {best['ll']:.2f}",
        f"Spectral radius A           : {best['sr']:.4f}",
        f"Transition noise Q mean     : {np.diag(best['Q']).mean():.4f}",
        f"Emission noise R mean       : {np.diag(best['R']).mean():.4f}",
        "",
        "Outcome prediction:",
    ]

    if pred_summary is None:
        diag_lines.append("  Skipped: missing outcome, one class only, or insufficient class counts.")
        mean_auc = np.nan
        overall_auc = np.nan
    else:
        diag_lines += [
            f"  Conversations used         : {pred_summary['n_conversations_used']}",
            f"  CV folds                   : {pred_summary['n_splits']}",
            f"  Mean CV AUC                : {pred_summary['mean_cv_auc']:.4f}",
            f"  Overall AUC                : {pred_summary['overall_auc']:.4f}",
        ]
        mean_auc = pred_summary["mean_cv_auc"]
        overall_auc = pred_summary["overall_auc"]

    with open(dataset_out / "stage2_diagnostics.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(diag_lines))

    print("\n" + "=" * 70)
    print(f"STAGE 2 COMPLETE — {dataset_name}")
    print(f"  Best SSM k       : {best_k}")
    print(f"  Spectral radius  : {best['sr']:.4f}")
    if pred_summary is not None:
        print(f"  Mean CV AUC      : {mean_auc:.4f}")
        print(f"  Overall AUC      : {overall_auc:.4f}")
    else:
        print("  Outcome AUC      : skipped")
    print(f"  Files saved to   : {dataset_out}")
    print("=" * 70)

    return {
        "dataset_name": dataset_name,
        "turns": len(df),
        "conversations": len(sequences),
        "retained_sst_dimensions": p_emotion,
        "obs_dim": obs_dim,
        "best_k": best_k,
        "best_bic": bic_results[best_k],
        "spectral_radius": best["sr"],
        "mean_cv_auc": mean_auc,
        "overall_auc": overall_auc,
        "output_dir": str(dataset_out),
    }


# ----------------------------
# Main loop
# ----------------------------
stage2_summary_rows = []
dataset_names = load_dataset_list()

for dataset_name in dataset_names:
    try:
        summary = run_stage2_for_dataset(dataset_name)
        stage2_summary_rows.append(summary)
    except Exception as exc:
        print("\n" + "!" * 70)
        print(f"STAGE 2 FAILED FOR DATASET: {dataset_name}")
        print(str(exc))
        print("!" * 70)
        stage2_summary_rows.append({
            "dataset_name": dataset_name,
            "error": str(exc),
            "output_dir": str(OUTPUT_ROOT / dataset_name),
        })

stage2_manifest = pd.DataFrame(stage2_summary_rows)
stage2_manifest.to_csv(OUTPUT_ROOT / "stage2_manifest.csv", index=False)

print("\n" + "=" * 70)
print("ALL DATASETS COMPLETE")
print("Stage 2 manifest:", OUTPUT_ROOT / "stage2_manifest.csv")
print("=" * 70)
stage2_manifest


In [ ]:
# Optional: zip all Stage 2 outputs for download
import shutil
from pathlib import Path
zip_path = shutil.make_archive('/content/optimal-nego/stage2_outputs', 'zip', '/content/optimal-nego/outputs')
print('Created:', zip_path)